In [109]:
from torch.utils.data import Dataset, DataLoader
import numpy as np
import torch
import pandas as pd
from scipy.spatial.distance import cdist
import scipy.sparse as sp
from tqdm import tqdm
from tqdm import trange

In [90]:
adata = sc.read_h5ad('../test.h5ad')
adata.obs

,tumor_gene_signature,stroma_immune_singnature_genes,t_gene_signature,T_means,leiden,X,Y,cell_type,tcr
s_016um_00052_00082-1,-13.163058,-110.718834,-7.980356,-0.173486,10,5307.678916618804,12449.199919631592,0,1
s_016um_00010_00367-1,-10.281478,-75.933578,-5.291239,-0.115027,0,6040.988190604934,16999.787108627097,1,0
s_016um_00163_00399-1,8.234170,488.635284,59.490974,1.293282,23,3599.0479211059037,17543.40584821326,0,1
s_016um_00238_00388-1,-4.404501,-17.279686,-7.980356,-0.173486,21,2396.76805388968,17382.86186754705,0,0
s_016um_00144_00175-1,4.236984,-45.988926,-4.228531,-0.091925,1,3855.484646416702,13956.137643030437,0,1
...,...,...,...,...,...,...,...,...,...
s_016um_00375_00231-1,-3.660754,23.671553,-7.980356,-0.173486,9,172.85211221427807,14900.346458407554,0,0
s_016um_00109_00223-1,-7.506592,43.515171,-4.085675,-0.088819,18,4425.694210888937,14716.500350242153,0,1
s_016um_00039_00175-1,-4.674540,-8.500046,-7.980356,-0.173486,12,5535.624672456852,13933.920353056192,0,0
s_016um_00037_00193-1,-13.163058,-103.328911,-7.980356,-0.173486,10,5571.488924089748,14221.434174253345,0,0


In [91]:
adata.var.index

Index(['ID1', 'CKMT2', 'MT-ND5', 'DNTTIP1', 'EIF4A3', 'ZFAND1', 'TSPO', 'OGT',
       'SUCLG2', 'ADI1',
       ...
       'PYGB', 'PHGR1', 'ELF3', 'CEACAM6', 'KLF5', 'CD24', 'ST14', 'CEACAM5',
       'MUC12', 'EPCAM'],
      dtype='object', length=500)

In [114]:
from torch.utils.data import Dataset, DataLoader
import numpy as np
import torch
import pandas as pd
from scipy.spatial.distance import cdist
import scipy.sparse as sp
from tqdm import trange

class BagsDataset(Dataset):
    def __init__(self, adata, radius=50, output_csv='bags.csv'):
        self.bags = self.create_bags(adata, radius, output_csv)

    def __len__(self):
        return len(self.bags)

    def __getitem__(self, idx):
        bag = self.bags[idx]
        distances = torch.tensor(bag['distances'], dtype=torch.float32)
        gene_expression = bag['gene_expression']
        if sp.issparse(gene_expression):
            gene_expression = torch.tensor(gene_expression.todense(), dtype=torch.float32)
        else:
            gene_expression = torch.tensor(gene_expression, dtype=torch.float32)
        label = torch.tensor(bag['label'], dtype=torch.float32)
        return distances, gene_expression, label

    def create_bags(self, adata, radius, output_csv):
        spatial_coords_x = adata.obs['X'].astype(float)
        spatial_coords_y = adata.obs['Y'].astype(float)
        spatial_coords = np.array(list(zip(spatial_coords_x, spatial_coords_y)))
        gene_expression = adata.X
        labels = adata.obs['tcr'].values
        cell_types = adata.obs['cell_type'].values
        barcodes = adata.obs.index.values

        bags = {}
        csv_data = []
        filtered_count = 0
        no_neighbors_count = 0
        bag_id = 0  # Initialize bag_id to ensure continuous IDs starting from 0

        for i in trange(len(spatial_coords), desc="Creating Bags", ncols=100, bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}{postfix}]"):
            if cell_types[i] == 0:
                continue  # Skip if the cell type is 0

            filtered_count += 1
            dist_matrix_row = cdist([spatial_coords[i]], spatial_coords, metric='euclidean')[0]
            in_circle = np.where(dist_matrix_row <= radius)[0]
            in_circle = [idx for idx in in_circle if cell_types[idx] != 0]  # Filter based on cell type
            if len(in_circle) == 0:
                no_neighbors_count += 1
                continue  # Skip if no instances meet the criteria

            gene_data = gene_expression[in_circle]
            distances = np.asmatrix(dist_matrix_row[in_circle].reshape(-1, 1), dtype=np.float32)

            bags[bag_id] = {
                'distances': distances,
                'gene_expression': gene_data,
                'label': labels[i]
            }

            bag_barcodes = barcodes[in_circle]
            for barcode in bag_barcodes:
                csv_data.append([bag_id, barcode, labels[i]])

            bag_id += 1  # Increment bag_id for the next bag

        total_bags = len(bags)
        avg_instances_per_bag = sum(len(bags[i]['gene_expression']) for i in bags) / total_bags
        print(f"Total bags created: {total_bags}")
        print(f"Average instances per bag: {avg_instances_per_bag:.0f}")

        return bags

def custom_collate_fn(batch):
    distances, gene_expressions, labels = zip(*batch)
    distances = [torch.tensor(np.array(d), dtype=torch.float32) for d in distances]
    gene_expressions_tensors = []
    for g in gene_expressions:
        if sp.issparse(g):
            gene_expressions_tensors.append(torch.tensor(g.todense(), dtype=torch.float32))
        else:
            gene_expressions_tensors.append(g.clone().detach().float())
    labels = torch.tensor(labels, dtype=torch.float32)
    return distances, gene_expressions_tensors, labels


In [116]:
dataset = BagsDataset(adata, radius= 20)
dataloader = DataLoader(dataset, batch_size=1, collate_fn=custom_collate_fn)

Creating Bags: 100%|██████████████████████████████████████| 137051/137051 [00:35<00:00, 3826.82it/s]


Total bags created: 65742
Average instances per bag: 4


In [117]:
len(adata.obs[adata.obs['cell_type'] == 1])

65742

In [118]:
len(dataloader)

65742

In [119]:
distances, gene_expressions, labels = next(iter(dataloader))

In [120]:
first_bag = dataset.bags[0]

In [121]:
first_bag['gene_expression'].shape

(4, 500)

In [122]:
dataloader.dataset.bags[2]['gene_expression']

array([[-0.6111422 , -0.37874946,  0.7928396 , ..., -1.1006578 ,
        -1.058933  , -1.0542383 ],
       [-0.6111422 , -0.37874946, -0.94990546, ..., -1.1006578 ,
        -1.058933  , -1.0542383 ],
       [-0.6111422 , -0.37874946, -0.94990546, ..., -1.1006578 ,
        -1.058933  , -1.0542383 ],
       [-0.6111422 , -0.37874946, -0.94990546, ..., -1.1006578 ,
        -1.058933  , -1.0542383 ]], dtype=float32)

In [123]:
dataloader.dataset.bags[2]['distances']

matrix([[ 0.      ],
        [16.00408 ],
        [16.004028],
        [15.994598]], dtype=float32)

In [124]:
dataloader.dataset.bags[0]['label']

0

In [125]:
dataset.__getitem__(2)

(tensor([[ 0.0000],
         [16.0041],
         [16.0040],
         [15.9946]]),
 tensor([[-0.6111, -0.3787,  0.7928,  ..., -1.1007, -1.0589, -1.0542],
         [-0.6111, -0.3787, -0.9499,  ..., -1.1007, -1.0589, -1.0542],
         [-0.6111, -0.3787, -0.9499,  ..., -1.1007, -1.0589, -1.0542],
         [-0.6111, -0.3787, -0.9499,  ..., -1.1007, -1.0589, -1.0542]]),
 tensor(1.))

process the visium H5AD

In [3]:
adata = sc.read_h5ad('/project/DPDS/Wang_lab/s439765/spatial_tcr/test/HumanColorectalCancer/HumanColorectalCancer.h5ad')

In [7]:
adata.obs['t_gene_signature'].mean()

np.float32(-4.1684413e-07)

In [1]:
def predict(model, adata, binding_csv, device, radius=50, batch_size=1):
    model.eval()
    binding_aff = pd.read_csv(binding_csv, index_col=0)
    dataset = BagsDataset(adata, binding_aff, radius)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_fn)
    
    predictions = []

    with torch.no_grad():
        with tqdm(dataloader, unit="batch") as tepoch:
            for distances, gene_expressions, affinity_data, _ in tepoch:
                tepoch.set_description("Predicting")
                
                distances = [d.to(device) for d in distances]
                gene_expressions = [g.to(device) for g in gene_expressions]
                affinity_data = [a.to(device) for a in affinity_data]

                instance_outputs = []
                for dist, gene_exp, aff_data in zip(distances, gene_expressions, affinity_data):
                    output = model(dist, gene_exp, aff_data)
                    instance_outputs.append(output)
                
                instance_outputs = torch.cat(instance_outputs, dim=0)
                bag_output = torch.mean(instance_outputs, dim=0, keepdim=True)
                predictions.append(bag_output.cpu().numpy().flatten())

    predictions = np.array(predictions)
    adata.obs['tcr_predict'] = predictions
    return adata

In [2]:
adata = predict(model, adata, binding_affinity)

NameError: name 'model' is not defined

In [ ]:


#can run with warnigns
from torch.utils.data import Dataset, DataLoader
import numpy as np
import torch
import pandas as pd
from scipy.spatial.distance import cdist
import scipy.sparse as sp

class BagsDataset(Dataset):
    def __init__(self, adata, radius=50, output_csv='bags.csv'):
        self.bags = self.create_bags(adata, radius, output_csv)

    def __len__(self):
        return len(self.bags)

    def __getitem__(self, idx):
        bag = self.bags[idx]
        distances = torch.tensor(bag['distances'], dtype=torch.float32)
        gene_expression = bag['gene_expression']
        if sp.issparse(gene_expression):
            gene_expression = torch.tensor(gene_expression.todense(), dtype=torch.float32)
        else:
            gene_expression = torch.tensor(gene_expression, dtype=torch.float32)
        label = torch.tensor(bag['label'], dtype=torch.float32)
        return distances, gene_expression, label

    def create_bags(self, adata, radius, output_csv):
        spatial_coords_x = adata.obs['X'].astype(float)
        spatial_coords_y = adata.obs['Y'].astype(float)
        spatial_coords = np.array(list(zip(spatial_coords_x, spatial_coords_y)))
        gene_expression = adata.X
        labels = adata.obs['tcr'].values
        cell_types = adata.obs['cell_type'].values
        barcodes = adata.obs.index.values

        bags = {}
        csv_data = []
        filtered_count = 0
        no_neighbors_count = 0
        bag_id = 0  # Initialize bag_id to ensure continuous IDs starting from 0

        for i in range(len(spatial_coords)):
            if cell_types[i] == 0:
                continue  # Skip if the cell type is 0

            filtered_count += 1
            dist_matrix_row = cdist([spatial_coords[i]], spatial_coords, metric='euclidean')[0]
            in_circle = np.where(dist_matrix_row <= radius)[0]
            in_circle = [idx for idx in in_circle if cell_types[idx] != 0]  # Filter based on cell type
            if len(in_circle) == 0:
                no_neighbors_count += 1
                continue  # Skip if no instances meet the criteria

            gene_data = gene_expression[in_circle]
            distances = np.asmatrix(dist_matrix_row[in_circle].reshape(-1, 1), dtype=np.float32)

            bags[bag_id] = {
                'distances': distances,
                'gene_expression': gene_data,
                'label': labels[i]
            }

            bag_barcodes = barcodes[in_circle]
            for barcode in bag_barcodes:
                csv_data.append([bag_id, barcode, labels[i]])

            print(f"Bag {bag_id} has {gene_data.shape[0]} instances")
            bag_id += 1  # Increment bag_id for the next bag

        print(f"Total points filtered by cell type: {filtered_count}")
        print(f"Total bags created: {len(bags)}")
        print(f"Total points with no neighbors: {no_neighbors_count}")

        df = pd.DataFrame(csv_data, columns=['bag_id', 'barcode', 'label'])
        df.to_csv(output_csv, index=False)
        return bags

def custom_collate_fn(batch):
    distances, gene_expressions, labels = zip(*batch)
    distances = [torch.tensor(np.array(d), dtype=torch.float32) for d in distances]
    gene_expressions_tensors = []
    for g in gene_expressions:
        if sp.issparse(g):
            gene_expressions_tensors.append(torch.tensor(g.todense(), dtype=torch.float32))
        else:
            gene_expressions_tensors.append(torch.tensor(g, dtype=torch.float32))
    labels = torch.tensor(labels, dtype=torch.float32)
    return distances, gene_expressions_tensors, labels